<a href="https://colab.research.google.com/github/weaamasad99/CloudComputingWolf/blob/main/Wolf_Index.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
# @title  Setup & Imports
# Install required libraries and import modules
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict
import nltk
from nltk.stem import PorterStemmer

# Download basic NLTK data required for stemming (if not already cached in Colab)
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [42]:
# @title  Configuration & Global Variables
#adding the DB
FIREBASE_URL = "https://lemonpulse-index-default-rtdb.europe-west1.firebasedatabase.app/lemon_disease_index.json"

# The 5 academic articles related to Citrus / Lemon diseases
ARTICLE_URLS = [
    "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0316081",
    "https://link.springer.com/article/10.1007/s13593-014-0246-1",
    "https://link.springer.com/article/10.1007/BF02215619",
    "https://ieeexplore.ieee.org/document/9481921",
    "https://www.techscience.com/cmc/v66n1/40458"
]

In [43]:
# @title  Text Processing Functions

def fetch_and_extract_content(url):
    """Fetch an article page and extract abstract/introduction text."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 11.0; Win64; x64) AppleWebKit/537.36"
    }
    content = ""
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")

            # Look for common abstract or introduction classes/tags
            for selector in ['div.abstract', 'section.Abstract', 'div.introduction', 'p']:
                elements = soup.select(selector)
                for element in elements[:15]: # Limit to first few paragraphs to get core content
                    text = element.get_text(strip=True)
                    if text:
                        content += " " + text
        return content
    except Exception as e:
        print(f"Failed to fetch {url}: {e}")
        return None

def get_stop_words():
      """Return an extended stop-word list to filter out noise."""
      basic_english = {
          "the", "is", "at", "which", "on", "and", "of", "to", "in", "a", "for", "with",
          "as", "by", "an", "that", "from", "this", "it", "are", "was", "were", "have",
          "has", "had", "but", "not", "can", "may", "will", "would", "could", "your", "they",
          "them", "their", "these", "those", "such", "very", "much", "many", "some",
          "we", "our", "all", "its", "also", "or", "if", "than", "then", "like", "into", "about",
          "how", "what", "when", "where", "who", "why", "any", "no", "only", "other", "so","plo", "sci", "too"
      }

      # (Academic & Research) stop words
      academic_terms = {
          "study", "paper", "research", "method", "methods", "result", "results",
          "analysis", "show", "use", "used", "using", "based", "data", "system", "approaches",
          "article", "articl", "response", "respons", "conclusion", "introduction", "abstract",
          "figure", "table", "section", "journal", "review", "literature", "proposed",
          "performance", "experiment", "model", "equation", "fig", "et", "al"
      }

      # General Noise & Time) stop words
      general_noise = {
          "new", "different", "important", "more", "etc", "year", "years", "high", "low",
          "large", "small", "good", "better", "modern", "well", "however", "thus", "therefore",
          "furthermore", "time", "day", "month", "first", "second", "one", "two", "three",
          "number", "value", "significant", "increase", "decrease", "within", "between"
      }

      return basic_english | academic_terms | general_noise

def clean_text(text):
    """Tokenize text, remove stop-words, apply Porter Stemmer, and return cleaned terms."""
    if not text:
        return []

    # Extract alphabetic words only, length >= 3
    words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    stop_words = get_stop_words()
    stemmer = PorterStemmer()

    cleaned_words = []
    for word in words:
        if word not in stop_words:
            stemmed = stemmer.stem(word)
            if stemmed not in stop_words and len(stemmed) >= 3:
                cleaned_words.append(stemmed)

    return cleaned_words

In [44]:
# @title TextRank & Indexing Logic
# TextRank Algorithm and Inverted Index Builders

def build_word_graph(words, window_size=4):
    """Build a co-occurrence graph for the TextRank algorithm."""
    graph = defaultdict(lambda: defaultdict(float))

    # Iterate through all words to establish connections based on proximity
    for i, word in enumerate(words):
        start = max(0, i - window_size)
        end = min(len(words), i + window_size + 1)

        for j in range(start, end):
            if i != j:
                neighbor = words[j]
                distance = abs(i - j)
                # Words that are closer to each other get a stronger weight (1/distance)
                graph[word][neighbor] += 1.0 / distance
    return graph

def calculate_textrank_scores(graph, damping=0.85, iterations=20):
    """Compute importance scores for each word using TextRank."""
    words = list(graph.keys())
    if not words:
        return {}

    # Initialize all nodes (words) with a base score of 1.0
    scores = {word: 1.0 for word in words}

    # Run power-iteration to converge the scores (similar to PageRank)
    for _ in range(iterations):
        new_scores = {}
        for word in words:
            # Base probability of jumping to a random node
            score = 1.0 - damping

            # Add distributed score from all connected neighbors
            for neighbor, connection_strength in graph[word].items():
                neighbor_total_connections = sum(graph[neighbor].values())
                if neighbor_total_connections > 0:
                    contribution = (connection_strength / neighbor_total_connections) * scores[neighbor]
                    score += damping * contribution
            new_scores[word] = score

        # Update scores for the next iteration
        scores = new_scores
    return scores

def count_word_frequencies(documents):
    """Create the inverted index mapping: term -> {doc_id: frequency_count}."""
    word_doc_counts = defaultdict(lambda: defaultdict(int))

    # Process each document and count term occurrences
    for doc_id, document in enumerate(documents, 1):
        words = clean_text(document)
        for word in words:
            word_doc_counts[word][doc_id] += 1

    return word_doc_counts

def extract_top_keywords(documents, top_k=30):
    """Combine documents, run TextRank, and return the top K keywords."""
    combined_text = " ".join(documents)
    words = clean_text(combined_text)

    # Fallback if the extracted content is extremely short
    if len(words) < 10:
        return words[:top_k]

    # Build the graph and calculate word importance
    graph = build_word_graph(words)
    scores = calculate_textrank_scores(graph)

    # Sort words by their TextRank score in descending order
    ranked_words = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Return only the string values of the top 'K' words
    return [word for word, score in ranked_words[:top_k]]

In [45]:
# @title Execution & Firebase Upload
# Main Execution Pipeline

def create_firebase_payload(top_keywords, word_doc_counts):
    """Format the inverted index into a JSON-friendly structure for Firebase."""
    firebase_data = {}

    for keyword in top_keywords:
        # Initialize the schema for each specific keyword
        firebase_data[keyword] = {
            "term": keyword,
            "DocsIds": {}
        }

        # Map document IDs to their term frequency.
        # Note: Firebase requires dictionary keys to be strings.
        for doc_id, count in word_doc_counts[keyword].items():
            firebase_data[keyword]["DocsIds"][str(doc_id)] = count

    return firebase_data

def upload_to_firebase(data, url):
    """Upload the final index to Firebase via PUT request."""
    try:
        # Using PUT will overwrite any existing data at this specific database node
        response = requests.put(url, json=data)
        if response.status_code in [200, 201]:
            print("\n[SUCCESS] Inverted Index uploaded to Firebase successfully!")
        else:
            print(f"\n[ERROR] Upload failed. Status Code: {response.status_code}")
    except Exception as e:
        print(f"\n[ERROR] Firebase connection error: {e}")


# Run Pipeline
documents = []
print("Fetching and processing articles...")

# Step 1: Fetch and clean content from all URLs
for i, url in enumerate(ARTICLE_URLS, 1):
    content = fetch_and_extract_content(url)
    # Ensure we actually scraped meaningful text (more than 50 chars)
    if content and len(content) > 50:
        documents.append(content)
        print(f" - Document {i} processed successfully.")
    else:
        print(f" - Document {i} could not be parsed (might be blocked/paywalled), skipping.")

# Step 2: If we have valid documents, build the index
if documents:
    print(f"\nExtracting keywords using TextRank from {len(documents)} documents...")

    # Extract keywords and calculate frequencies
    keywords = extract_top_keywords(documents, top_k=30)
    word_doc_counts = count_word_frequencies(documents)

    # Format the data for NoSQL DB storage
    firebase_data = create_firebase_payload(keywords, word_doc_counts)

    # Display the final results to the console
    print("\n--- TOP TEXTRANK KEYWORDS FOUND ---")
    for i, keyword in enumerate(keywords, 1):
        doc_distribution = ", ".join([f"Doc {doc_id}:{count}" for doc_id, count in firebase_data[keyword]["DocsIds"].items()])
        print(f"{i:2d}. {keyword} --> {doc_distribution}")

    # Step 3: Push the data to the cloud
    upload_to_firebase(firebase_data, FIREBASE_URL)
else:
    print("\n[FAILED] No content was extracted from the URLs.")

Fetching and processing articles...
 - Document 1 processed successfully.
 - Document 2 processed successfully.
 - Document 3 processed successfully.
 - Document 4 processed successfully.
 - Document 5 processed successfully.

Extracting keywords using TextRank from 5 documents...

--- TOP TEXTRANK KEYWORDS FOUND ---
 1. diseas --> Doc 1:4, Doc 2:6, Doc 5:2
 2. fruit --> Doc 1:2, Doc 3:8
 3. irrig --> Doc 3:8
 4. citru --> Doc 1:3, Doc 3:4
 5. treatment --> Doc 3:8
 6. plant --> Doc 2:5, Doc 3:2
 7. effect --> Doc 1:1, Doc 2:3, Doc 3:2
 8. water --> Doc 3:6
 9. detect --> Doc 1:1, Doc 2:4, Doc 5:1
10. technolog --> Doc 1:2, Doc 2:1, Doc 4:1, Doc 5:2
11. yield --> Doc 1:1, Doc 3:5
12. univers --> Doc 1:3, Doc 5:3
13. access --> Doc 1:1, Doc 2:2, Doc 3:2, Doc 5:1
14. lemon --> Doc 1:1, Doc 3:4
15. learn --> Doc 1:3, Doc 5:1
16. comput --> Doc 1:3, Doc 5:2
17. period --> Doc 3:5
18. growth --> Doc 3:5
19. dna --> Doc 2:4
20. leaf --> Doc 3:2, Doc 5:2
21. agricultur --> Doc 1:2, Doc 2:2
22